In [ ]:
# Wave clear / item path (Summoner's Rift 5v5)

This notebook uses `LoLPerfmon.sim`: load patch data, run `simulate`, optional `best_item_order_exhaustive`.

**Automated checks:** run the cell below (`run_validation`) to execute the same assertions as `LoLPerfmon/tests/` and get a JSON-friendly report—suitable for AI agents without pytest.

### Run all simulation checks (no pytest)

`LoLPerfmon.validation_checks` defines one function per check, `run_validation()` for a list of `{id, ok, error}`, and `format_report(..., style="json")` for machine-readable output.

In [ ]:
import sys
import json
import os
from pathlib import Path

cwd = Path.cwd()
for root in (cwd, cwd.parent):
    if (root / "LoLPerfmon" / "sim").is_dir():
        sys.path.insert(0, str(root))
        break
else:
    raise RuntimeError("Could not find LoLPerfmon/sim; cwd should be repo root or LoLPerfmon/")

from LoLPerfmon.validation_checks import format_report, run_validation, validation_summary

# offline=True: no network (computed SR defaults). Set LOLPERFMON_OFFLINE=0 to try Data Dragon first.
offline = os.environ.get("LOLPERFMON_OFFLINE", "1").lower() in ("1", "true", "yes")
report = run_validation(data_dir=None, offline=offline)
summary = validation_summary(report)
print(format_report(report, style="text"))
print()
print("JSON summary for agents:")
print(json.dumps(summary, indent=2))
assert summary["all_ok"], f"{summary['failed']} check(s) failed: {summary['failures']}"

In [ ]:
### Example: one simulation (Lux, lane, sample buy order)

Requires the same `sys.path` setup as above; re-run the next cell alone only if that cell was already executed.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd()
for root in (cwd, cwd.parent):
    if (root / "LoLPerfmon" / "sim").is_dir():
        sys.path.insert(0, str(root))
        break
else:
    raise RuntimeError("Could not find LoLPerfmon/sim; run from repo root or LoLPerfmon/")

from LoLPerfmon.sim import FarmMode, PurchasePolicy, get_game_bundle, simulate
from LoLPerfmon.sim.test_champion_ids import primary_champion_id, two_cheapest_item_ids

offline = os.environ.get("LOLPERFMON_OFFLINE", "1").lower() in ("1", "true", "yes")
data = get_game_bundle(offline=offline)
cid = primary_champion_id(data)
a, b = two_cheapest_item_ids(data)
res = simulate(
    data,
    cid,
    FarmMode.LANE,
    PurchasePolicy(buy_order=(a, b)),
    t_max=900.0,
)
res.final_gold, res.final_level, res.final_inventory, res.total_farm_gold, res.total_passive_gold